# MethylLlama — Embedding Analysis

Visualise the CLS embeddings extracted from the pretrained encoder using UMAP.
We show two views as a combined figure:

- **(a) Age gradient** — does the 256-D CLS space encode a continuous age signal?
- **(b) Tissue structure** — does it separate tissue types without any supervision?

These reproduce the UMAP figure from the thesis on the 500-sample public demo subset.

In [ ]:
import sys
from pathlib import Path

repo_root = Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import os
import torch
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load model and demo data

In [ ]:
from huggingface_hub import hf_hub_download

from bmfm_methylation.llama.model import MethylLlamaConfig
from bmfm_methylation.llama.wced_llama import WCEDLlamaModule

# Patch torch.load for PyTorch >= 2.6 (weights_only=True breaks Lightning ckpts)
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

# Download checkpoint
ckpt_path = hf_hub_download(
    repo_id="netanelazran1/MethylLlama",
    filename="pretrain_llama_epoch98_val0.0059.ckpt",
)
ckpt = torch.load(ckpt_path, map_location="cpu")

vocab_size = ckpt["state_dict"]["encoder.embeddings.cpg_sites_embeddings.weight"].shape[0]

model_config = MethylLlamaConfig(
    hidden_size=256, num_hidden_layers=4, num_attention_heads=4,
    intermediate_size=320, vocab_size=vocab_size,
)
module = WCEDLlamaModule(
    model_config=model_config,
    vocab_size=ckpt["hyper_parameters"]["vocab_size"],
)
module.load_state_dict(ckpt["state_dict"])
encoder = module.encoder.to(DEVICE).eval()
print("Model loaded.")

In [ ]:
# Load demo data
adata = ad.read_h5ad("data/methylllama_demo_500samples.h5ad")
cpg_names = list(adata.var_names)
print(f"Demo: {adata.shape[0]} samples × {adata.shape[1]} CpGs")

# Build vocab dict directly from the pretrained 49k-CpG vocabulary
vocab_path = hf_hub_download(
    repo_id="netanelazran1/MethylLlama",
    filename="tokenizer/cpg_sites/vocab.txt",
)
cpg_vocab = {}
with open(vocab_path) as f:
    for idx, line in enumerate(f):
        cpg_vocab[line.strip()] = idx

cls_id = cpg_vocab["[CLS]"]
unk_id = cpg_vocab["[UNK]"]

demo_ids = [cpg_vocab.get(c, unk_id) for c in cpg_names]
n_known  = sum(1 for i in demo_ids if i != unk_id)
print(f"Vocabulary: {len(cpg_vocab)} CpG tokens")
print(f"Demo CpGs in pretrained vocab: {n_known}/{len(cpg_names)} ({100*n_known/len(cpg_names):.1f}%)")

## 2. Extract CLS embeddings for all 500 samples

In [ ]:
def extract_cls(adata, encoder, cpg_vocab, n_cpg=8000, batch_size=16, seed=42):
    """Extract pooler_output (CLS) for all samples using known CpGs only."""
    rng    = np.random.default_rng(seed)
    unk_id = cpg_vocab["[UNK]"]
    cls_id = cpg_vocab["[CLS]"]

    X = adata.X
    if hasattr(X, "toarray"):
        X = X.toarray()
    X = X.astype(np.float32)

    names   = list(adata.var_names)
    all_ids = np.array([cpg_vocab.get(c, unk_id) for c in names])

    # Use only CpGs present in the pretrained vocab
    known_mask = all_ids != unk_id
    known_idx  = np.where(known_mask)[0]
    n_cpg      = min(n_cpg, len(known_idx))

    sel_local     = rng.choice(len(known_idx), size=n_cpg, replace=False)
    sel_idx       = known_idx[sel_local]
    sel_token_ids = all_ids[sel_idx].tolist()
    L             = n_cpg + 1

    all_cls = []
    n = adata.shape[0]

    with torch.no_grad():
        for start in range(0, n, batch_size):
            end   = min(start + batch_size, n)
            betas = X[start:end, sel_idx].copy()
            betas[np.isnan(betas)] = 0.5

            cpg_ids   = torch.tensor([[cls_id] + sel_token_ids] * (end - start),
                                     dtype=torch.float32)
            beta_vals = torch.cat([
                torch.full((end - start, 1), -2.0),
                torch.tensor(betas, dtype=torch.float32)
            ], dim=1)
            input_ids = torch.stack([cpg_ids, beta_vals], dim=1).to(DEVICE)  # [B,2,L]
            mask      = torch.ones(end - start, L, dtype=torch.long).to(DEVICE)

            out = encoder(input_ids=input_ids, attention_mask=mask)
            all_cls.append(out.pooler_output.cpu().numpy())

    return np.vstack(all_cls)


# Use up to 8000 known CpGs for maximum information
cls_emb = extract_cls(adata, encoder, cpg_vocab, n_cpg=8000)
print(f"CLS embeddings: {cls_emb.shape}")

## 3. UMAP

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=42)
emb_2d  = reducer.fit_transform(cls_emb)
print(f"UMAP output: {emb_2d.shape}")

## 3. Two-panel UMAP figure  (publication style)

In [ ]:
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "svg.fonttype": "none"})

ages    = adata.obs["age"].astype(float).values
tissues = np.array(adata.obs["tissue_type"].values)

# Nature/Cell-style curated palette
PALETTE = {
    "blood cd14+ monocyte":         "#E64B35",
    "blood cord":                   "#4DBBD5",
    "blood leukocyte":              "#F39B7F",
    "blood wbc":                    "#3C5488",
    "blood whole":                  "#00A087",
    "brain cerebellum":             "#7E6148",
    "brain frontal cortex":         "#B09C85",
    "brain superior temporal gyrus":"#91D1C2",
    "breast":                       "#DC0000",
    "colon":                        "#ABA300",
    "kidney":                       "#00BFC4",
    "liver":                        "#8491B4",
    "lung":                         "#C77CFF",
    "prostate":                     "#FF61CC",
    "thyroid":                      "#00BE67",
    "uterus":                       "#CD9600",
}
OFFSETS = {
    "blood cd14+ monocyte":          (-0.3,  0.5),
    "blood cord":                    ( 0.5,  0.0),
    "blood leukocyte":               ( 0.3, -0.5),
    "blood wbc":                     ( 0.0,  0.4),
    "brain superior temporal gyrus": (-0.2,  0.3),
    "brain frontal cortex":          (-0.2, -0.3),
}
unique_tis = sorted(set(tissues))

def _add_arrows(ax):
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    xl, xr = ax.get_xlim(); yb, yt = ax.get_ylim()
    sx = (xr-xl)*0.11; sy = (yt-yb)*0.10
    ox = xl+(xr-xl)*0.01; oy = yb+(yt-yb)*0.02
    kw = dict(arrowstyle="->", color="#555555", lw=1.2, mutation_scale=10, zorder=6)
    ax.add_patch(FancyArrowPatch((ox,oy),(ox+sx,oy),**kw))
    ax.add_patch(FancyArrowPatch((ox,oy),(ox,oy+sy),**kw))
    ax.text(ox+sx/2, oy-(yt-yb)*0.025, "UMAP 1", fontsize=7.5,
            ha="center", va="top", color="#555555")
    ax.text(ox-(xr-xl)*0.012, oy+sy/2, "UMAP 2", fontsize=7.5,
            ha="right", va="center", color="#555555", rotation=90)

fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
fig.patch.set_facecolor("white")

# Panel a — age
ax = axes[0]; ax.set_facecolor("white")
sc = ax.scatter(emb_2d[:,0], emb_2d[:,1], c=ages, cmap="plasma",
                s=22, alpha=0.85, linewidths=0, zorder=2)
cb = fig.colorbar(sc, ax=ax, shrink=0.6, aspect=18, pad=0.02)
cb.set_label("Chronological age (yr)", fontsize=8.5, labelpad=6)
cb.ax.tick_params(labelsize=8); cb.outline.set_visible(False)
_add_arrows(ax)
ax.text(0.02, 0.98, "a", transform=ax.transAxes,
        fontsize=14, fontweight="bold", va="top", ha="left")
ax.set_title("Coloured by chronological age", fontsize=9.5, pad=6, color="#333333")

# Panel b — tissue
ax = axes[1]; ax.set_facecolor("white")
for t in unique_tis:
    mask = tissues == t
    ax.scatter(emb_2d[mask,0], emb_2d[mask,1],
               c=PALETTE.get(t, "#aaaaaa"), s=22, alpha=0.82, linewidths=0, zorder=2)
for t in unique_tis:
    mask = tissues == t
    if mask.sum() < 3: continue
    cx = emb_2d[mask,0].mean() + OFFSETS.get(t,(0,0))[0]
    cy = emb_2d[mask,1].mean() + OFFSETS.get(t,(0,0))[1]
    txt = ax.text(cx, cy, t, fontsize=6.2, ha="center", va="center",
                  color="#1a1a1a", fontweight="bold", zorder=5)
    txt.set_path_effects([pe.withStroke(linewidth=2.8, foreground="white")])
_add_arrows(ax)
ax.text(0.02, 0.98, "b", transform=ax.transAxes,
        fontsize=14, fontweight="bold", va="top", ha="left")
ax.set_title("Coloured by tissue type", fontsize=9.5, pad=6, color="#333333")

fig.suptitle("MethylLlama CLS embeddings — UMAP  (pretrained encoder, no fine-tuning)",
             fontsize=10.5, y=1.01, color="#222222")
plt.tight_layout(w_pad=3)
plt.savefig("umap_figure.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: umap_figure.png")

In [ ]:
from scipy.stats import spearmanr, pearsonr

rho, _  = spearmanr(ages, emb_2d[:, 0])
r,   _  = pearsonr(ages, emb_2d[:, 0])
print(f"UMAP axis 1 vs age:  Spearman ρ = {rho:.3f}  |  Pearson r = {r:.3f}")
print()
print("The pretrained CLS embedding captures a strong age signal")
print("even without any fine-tuning on age labels.")